# 5. Embeddings

`embed()` returns the representation the output head would have received. It is the
natural thing to cluster, to compare sequences with, or to feed to a different model.

Two things decide whether an embedding comparison means anything, and both are easy to
get wrong:

1. **Width is not fixed.** It is the body width for every released model, and would
   become the projection width if a projection were ever trained. Ask the model; never
   hard-code 768.
2. **Use cosine similarity, not Euclidean distance.** The body is a plain residual stack
   with no renormalisation after the residual add, so activation norm grows with depth.
   Any unnormalised distance inherits that growth. Cosine similarity is scale invariant
   and does not.

In [1]:
import torch

torch.set_num_threads(2)

import numpy as np
import parnet_tutorials as pt
from parnet.models import RBPNet

model = RBPNet.from_pretrained(pt.model_path('parnet-21m-penalty5'))
arch = model.parnet_config['architecture']

probe = model.embed(torch.zeros(1, 4, 600))
print('embedding tensor:', tuple(probe.shape), '(batch, channels, positions)')
print('width           :', probe.shape[1])
print('body width      :', arch['body_width'])
print('has projection  :', arch['use_projection'])

/home/user/projects/parnet-project/parnet--analyses--interpretation/.pixi/envs/parnet-dev-cu12/lib/python3.10/site-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv1d(


embedding tensor: (1, 768, 600) (batch, channels, positions)
width           : 768
body width      : 768
has projection  : False


## Three aggregations

`embed()` with no aggregation gives one vector per position. For a per-sequence vector
you reduce over positions, and the choice matters:

- `mean` uses the whole window, including the parts the model finds uninteresting
- `center` takes the middle position only, which is the position with the most context on
  both sides and the natural choice when the window is centred on a site of interest

In [2]:
rng = np.random.default_rng(0)
sequences = [''.join(rng.choice(list('ACGT'), size=600)) for _ in range(8)]

with torch.no_grad():
    full = model.embed_from_sequence(sequences)
    mean = model.embed_from_sequence(sequences, aggregation='mean')
    center = model.embed_from_sequence(sequences, aggregation='center')

for name, tensor in [('none', full), ('mean', mean), ('center', center)]:
    print(f'{name:8s} {tuple(tensor.shape)}')

/tmp/ipykernel_70565/1534557451.py:5: DeprecationWarning: embed_from_sequence is deprecated and will be removed at 1.0. Tokenize explicitly instead:
    encoded = RnaTokenizer()(sequences)
    model.embed(encoded['input_ids'])
Until it is removed this method still silently accepts a lowercase sequence as all-N and returns a confident prediction for it.
  full = model.embed_from_sequence(sequences)


/tmp/ipykernel_70565/1534557451.py:6: DeprecationWarning: embed_from_sequence is deprecated and will be removed at 1.0. Tokenize explicitly instead:
    encoded = RnaTokenizer()(sequences)
    model.embed(encoded['input_ids'])
Until it is removed this method still silently accepts a lowercase sequence as all-N and returns a confident prediction for it.
  mean = model.embed_from_sequence(sequences, aggregation='mean')


/tmp/ipykernel_70565/1534557451.py:7: DeprecationWarning: embed_from_sequence is deprecated and will be removed at 1.0. Tokenize explicitly instead:
    encoded = RnaTokenizer()(sequences)
    model.embed(encoded['input_ids'])
Until it is removed this method still silently accepts a lowercase sequence as all-N and returns a confident prediction for it.
  center = model.embed_from_sequence(sequences, aggregation='center')


none     (8, 768, 600)
mean     (8, 768)
center   (8, 768)


## Why not Euclidean distance

Measure the activation norm through the stack. It grows, because each residual block adds
its branch output to the running sum and nothing renormalises afterwards.

In [3]:
x = torch.stack([__import__('parnet.utils', fromlist=['sequence_to_onehot'])
                 .sequence_to_onehot(s) for s in sequences[:2]]).float()

norms = []
with torch.no_grad():
    h = model.stem(x)
    norms.append(('stem', h.norm(dim=1).mean().item()))
    for i, block in enumerate(model.body):
        h = block(h)
        norms.append((f'body.{i}', h.norm(dim=1).mean().item()))

for name, value in norms:
    print(f'{name:8s} mean per-position norm {value:7.3f}')

growth = norms[-1][1] / norms[0][1]
print(f'\nnorm grows {growth:.1f}x from stem to final block')
print('Any unnormalised distance on late-layer embeddings inherits this,')
print('independently of anything biological.')

stem     mean per-position norm   1.736
body.0   mean per-position norm   1.962
body.1   mean per-position norm   2.123
body.2   mean per-position norm   2.355
body.3   mean per-position norm   2.677
body.4   mean per-position norm   3.064
body.5   mean per-position norm   3.554
body.6   mean per-position norm   4.131
body.7   mean per-position norm   4.792
body.8   mean per-position norm   5.373
body.9   mean per-position norm   5.963
body.10  mean per-position norm   6.347
body.11  mean per-position norm   6.801

norm grows 3.9x from stem to final block
Any unnormalised distance on late-layer embeddings inherits this,
independently of anything biological.


## Comparing sequences

Build a small set with structure in it: several near-copies of two different motif
contexts, plus random sequence. Cosine similarity on the centre embedding should separate
them.

In [4]:
def with_motif(motif, n, seed):
    """n sequences, each random except for `motif` repeated around the centre."""
    local = np.random.default_rng(seed)
    out = []
    for _ in range(n):
        seq = list(local.choice(list('ACGT'), size=600))
        block = motif * (60 // len(motif))
        seq[270:270 + len(block)] = list(block)
        out.append(''.join(seq))
    return out


groups = {
    'UGCAUG (RBFOX-like)': with_motif('TGCATG', 6, 1),
    'ACUAAC (QKI-like)': with_motif('ACTAAC', 6, 2),
    'random': [''.join(np.random.default_rng(3 + i).choice(list('ACGT'), size=600)) for i in range(6)],
}

labels, allseq = [], []
for name, seqs in groups.items():
    labels += [name] * len(seqs)
    allseq += seqs

with torch.no_grad():
    vectors = model.embed_from_sequence(allseq, aggregation='center')

normalised = torch.nn.functional.normalize(vectors, dim=1)
similarity = (normalised @ normalised.T).numpy()
print('embeddings:', tuple(vectors.shape))

/tmp/ipykernel_70565/4058289461.py:25: DeprecationWarning: embed_from_sequence is deprecated and will be removed at 1.0. Tokenize explicitly instead:
    encoded = RnaTokenizer()(sequences)
    model.embed(encoded['input_ids'])
Until it is removed this method still silently accepts a lowercase sequence as all-N and returns a confident prediction for it.
  vectors = model.embed_from_sequence(allseq, aggregation='center')


embeddings: (18, 768)


In [5]:
import pandas as pd

labels_array = np.array(labels)
names = list(groups)
table = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        block = similarity[np.ix_(labels_array == a, labels_array == b)]
        if a == b:
            iu = np.triu_indices(block.shape[0], k=1)
            table.loc[a, b] = block[iu].mean()
        else:
            table.loc[a, b] = block.mean()

pd.set_option('display.width', 200)
print('mean cosine similarity, within and between groups:')
print(table.to_string(float_format=lambda v: f'{v:.4f}'))

within = np.mean([table.loc[n, n] for n in names[:2]])
between = table.loc[names[0], names[1]]
print(f'\nwithin motif groups {within:.4f}  vs  between them {between:.4f}')

mean cosine similarity, within and between groups:
                     UGCAUG (RBFOX-like)  ACUAAC (QKI-like)  random
UGCAUG (RBFOX-like)               0.9755             0.3782  0.5316
ACUAAC (QKI-like)                 0.3782             0.9917  0.3852
random                            0.5316             0.3852  0.7224

within motif groups 0.9836  vs  between them 0.3782


## Projecting to two dimensions

UMAP is the usual choice. It is an optional dependency here, so this falls back to PCA
rather than failing: the point of the notebook is the embedding, not the projector.

In [6]:
coords, method = None, None
try:
    import umap

    coords = umap.UMAP(n_neighbors=5, min_dist=0.3, random_state=0,
                       metric='cosine').fit_transform(vectors.numpy())
    method = 'UMAP (cosine)'
except ImportError:
    from sklearn.decomposition import PCA

    coords = PCA(n_components=2, random_state=0).fit_transform(normalised.numpy())
    method = 'PCA on L2-normalised embeddings (umap-learn not installed)'

print('projection:', method)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.5, 4.5))
for name in names:
    mask = labels_array == name
    ax.scatter(coords[mask, 0], coords[mask, 1], label=name, s=55)
ax.set_title(f'centre embeddings, {method}', fontsize=10)
ax.set_xlabel('component 1')
ax.set_ylabel('component 2')
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

results = pt.project_root() / 'results'
results.mkdir(exist_ok=True)
fig.savefig(results / '05_embedding_projection.png', dpi=120)
print('saved', results / '05_embedding_projection.png')

2026-10-28 09:59:57.224546: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-28 09:59:57.232927: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-28 09:59:57.244290: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-28 09:59:57.244321: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-28 09:59:57.252780: I tensorflow/core/platform/cpu_feature_gua

2026-10-28 09:59:57.933414: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


/home/user/projects/parnet-project/parnet--analyses--interpretation/.pixi/envs/parnet-dev-cu12/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


projection: UMAP (cosine)
saved /data/projects/lambosaur/parnet-tutorials/results/05_embedding_projection.png


## A caution about reading these plots

A projection of a handful of synthetic sequences is a sanity check, not a result. For a
real analysis use genuine peak regions, and pick RBPs with genuinely different motifs,
otherwise you are asking the embedding to separate things that are not separable.

From the k-mer work, the RBPs with reproducible, high-effect motifs are **EIF4G2, TAF15,
PCBP1, RBFOX2, FUS and EWSR1** on RBNS, and **QKI, U2AF2, CPEB4, PABPN1 and RBM5** on
RNAcompete. **TIA1 and HNRNPC sit below the window-selection noise floor** and are poor
choices for a demonstration.

## What to take away

1. Ask the model for its embedding width.
2. Use cosine similarity. Activation norm grows through the body (measured above on
   this model and this input), and any unnormalised metric picks that up as signal.
3. `center` is the right aggregation when the window is centred on a site of interest.
4. Choose RBPs whose motifs actually differ before concluding anything from a projection.